# PCA — Mathematical Formulation

*Course notes for **Math for Machine Learning**, C1 · W4 · L2 · V08c — "PCA: Mathematical Formulation" (DeepLearning.AI).*

One last, formal pass over the PCA algorithm in **standard notation**, tying together every formula from the lesson. The running example has **5 variables** reduced to **2 dimensions**, but the steps work for any sizes.

In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## The steps

**1. Arrange the data.** Put observations in rows, variables in columns — an $n \times d$ matrix $X$ (here $d = 5$). *(This is the matrix earlier called $A$; the standard name is $X$ since the variables are $x_1, \dots, x_5$.)*

**2. Center.** Subtract each column's mean, giving $X - \mu$.

**3. Covariance matrix.** A single matrix multiplication:
$$ C = \frac{1}{n-1}\,(X - \mu)^T (X - \mu) \quad (d \times d,\ \text{here } 5 \times 5). $$

**4. Eigen-decomposition.** Find the eigenvalues/eigenvectors of $C$ and **sort by eigenvalue**. To reduce to $k = 2$ dimensions, keep the **two largest** eigenvalue–eigenvector pairs.

**5. Projection matrix.** Stack the chosen eigenvectors (each **scaled to norm 1**) as columns of $V$ ($d \times k$).

**6. Project.** 
$$ X_{PCA} = (X - \mu)\,V \quad (n \times k). $$

The result $X_{PCA}$ has the same $n$ observations but only $k = 2$ columns — the projection onto the top principal components.

In [2]:
# A 5-variable dataset with n observations
rng = np.random.default_rng(4)
n = 300
latent = rng.normal(size=(n, 2))              # true 2-D structure
mix = rng.normal(size=(2, 5))
X = latent @ mix + rng.normal(scale=0.2, size=(n, 5)) + np.array([3, -1, 2, 0, 5])
print('Step 1 - data X:', X.shape, '(n observations x 5 variables)')

Step 1 - data X: (300, 5) (n observations x 5 variables)


In [3]:
# Step 2: center
mu = X.mean(axis=0)
Xc = X - mu
print('Step 2 - column means mu =', mu)
print('centered column means    =', Xc.mean(axis=0))

Step 2 - column means mu = [ 3.006 -0.968  2.024 -0.022  5.012]
centered column means    = [-0.  0. -0. -0. -0.]


In [4]:
# Step 3: covariance matrix  C = (1/(n-1)) (X-mu)^T (X-mu)
C = (Xc.T @ Xc) / (n - 1)
print('Step 3 - covariance matrix C (5x5):')
print(C)
print('matches np.cov:', np.allclose(C, np.cov(X.T, ddof=1)))

Step 3 - covariance matrix C (5x5):
[[ 0.723 -2.352 -0.539  0.596 -0.33 ]
 [-2.352  9.441  3.836 -3.197  2.297]
 [-0.539  3.836  3.44  -2.197  2.008]
 [ 0.596 -3.197 -2.197  1.557 -1.299]
 [-0.33   2.297  2.008 -1.299  1.222]]
matches np.cov: True


In [5]:
# Step 4: eigenvalues/eigenvectors, sorted by eigenvalue (largest first)
vals, vecs = np.linalg.eigh(C)               # symmetric -> real eigenvalues, orthonormal eigenvectors
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
print('Step 4 - eigenvalues (desc):', vals)
print('variance explained by each:', (vals / vals.sum()).round(3))

Step 4 - eigenvalues (desc): [13.929  2.342  0.039  0.038  0.035]
variance explained by each: [0.85  0.143 0.002 0.002 0.002]


In [6]:
# Step 5: projection matrix V = top-2 eigenvectors as columns (already unit norm from eigh)
k = 2
V = vecs[:, :k]
print('Step 5 - projection matrix V (5 x 2):')
print(V)
print('columns are unit norm:', np.allclose(np.linalg.norm(V, axis=0), 1))

Step 5 - projection matrix V (5 x 2):
[[-0.179  0.321]
 [ 0.798 -0.492]
 [ 0.414  0.666]
 [-0.314 -0.253]
 [ 0.247  0.384]]
columns are unit norm: True


In [7]:
# Step 6: project  X_PCA = (X - mu) V
X_pca = Xc @ V
print('Step 6 - X_PCA:', X_pca.shape, '(n observations x 2 principal components)')
print(X_pca[:5])
print()
print('variance retained by top 2 of 5 dims: {:.1%}'.format(vals[:k].sum() / vals.sum()))

Step 6 - X_PCA: (300, 2) (n observations x 2 principal components)
[[-2.089 -0.54 ]
 [ 5.202  1.972]
 [-5.368 -0.292]
 [-2.273 -0.146]
 [-5.934 -0.689]]

variance retained by top 2 of 5 dims: 99.3%


## As a reusable function

The whole procedure in one place — the canonical PCA reference:

In [8]:
def pca(X, k):
    mu = X.mean(axis=0)
    Xc = X - mu                                  # 2. center
    C = (Xc.T @ Xc) / (X.shape[0] - 1)           # 3. covariance
    vals, vecs = np.linalg.eigh(C)               # 4. eigen-decomposition
    order = np.argsort(vals)[::-1]               #    sort by eigenvalue (largest first)
    V = vecs[:, order][:, :k]                    # 5. top-k eigenvectors as columns
    return Xc @ V                                # 6. project

print('pca(X, 2) shape:', pca(X, 2).shape)
print('matches step-by-step result:', np.allclose(np.abs(pca(X, 2)), np.abs(X_pca)))

pca(X, 2) shape: (300, 2)
matches step-by-step result: True


## Summary — PCA in six steps

1. **Arrange** data as $X$ ($n \times d$: observations in rows, variables in columns).
2. **Center**: $X - \mu$ (subtract column means).
3. **Covariance**: $C = \frac{1}{n-1}(X-\mu)^T(X-\mu)$.
4. **Eigen-decompose** $C$ and sort by eigenvalue; keep the **$k$ largest** pairs.
5. **Projection matrix** $V$: the chosen unit-norm eigenvectors as columns ($d \times k$).
6. **Project**: $X_{PCA} = (X-\mu)\,V$ — an $n \times k$ dataset onto the top principal components.

That is the complete recipe for **dimensionality reduction with PCA**.